**PART 1**

**Task 1: Data Loading and Initial Inspection**

In [ ]:
import pandas as pd

url1 = "https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/Bank%20Churn%20Modelling.csv"
df1 = pd.read_csv(url1)

df2 = pd.read_csv("Bank_Churn_Modelling_extra_UPDATED_v2.csv")


# rows and columns
df1.shape
df2.shape

# data types
df1.dtypes
df2.dtypes

df1.columns.tolist()
df2.columns.tolist()

df1.head()
df2.head()



**Task 2: Consolidation**

In [ ]:
print("df1 rows:", len(df1))
print("df2 rows:", len(df2))

df_combined = pd.concat([df1, df2], ignore_index=True)
print("Combined rows:", len(df_combined))

**Task 3: Duplicate Analysis and Resolution**

In [ ]:
full_dupes = df_combined.duplicated()
print("Fully identical duplicates:", full_dupes.sum())

df_combined[df_combined.duplicated(keep=False)].sort_values("CustomerId")


id_dupes = df_combined[df_combined.duplicated(subset=["CustomerId"], keep=False)]
print("Rows sharing a CustomerId:", len(id_dupes))
id_dupes.sort_values("CustomerId")


# Fully identical rows
df_combined = df_combined.drop_duplicates()
print("After removing full duplicates:", len(df_combined))

# For partial duplicates
df_combined = df_combined.drop_duplicates(subset=["CustomerId"], keep="first")
print("After removing partial duplicates:", len(df_combined))


**Task 4: Finding and Filling Missing Values **

In [ ]:
df_combined.isnull().sum()
df_combined.isnull().sum() / len(df_combined) * 100

print("Columns in df_combined:", df_combined.columns.tolist())

# Numeric columns (median)
df_combined["Age"] = df_combined["Age"].fillna(df_combined["Age"].median())
df_combined["CreditScore"] = df_combined["CreditScore"].fillna(df_combined["CreditScore"].median())
df_combined["Balance"] = df_combined["Balance"].fillna(df_combined["Balance"].median())
df_combined["Estimated Salary"] = df_combined["Estimated Salary"].fillna(df_combined["Estimated Salary"].median())

# Categorical columns
df_combined["Gender"] = df_combined["Gender"].fillna(df_combined["Gender"].mode()[0])
df_combined["Geography"] = df_combined["Geography"].fillna(df_combined["Geography"].mode()[0])



df_combined.isnull().sum()

**Task 5: Data Type and Structural Refinements**

In [ ]:
df_combined.dtypes

# Change binary/categorical columns
df_combined["Gender"] = df_combined["Gender"].astype("category")
df_combined["Geography"] = df_combined["Geography"].astype("category")
df_combined["Has Credit Card"] = df_combined["Has Credit Card"].astype("category")
df_combined["Is Active Member"] = df_combined["Is Active Member"].astype("category")
df_combined["Churn"] = df_combined["Churn"].astype("category")

# Drop irrelevant columns
df_combined.drop(columns=["CustomerId", "Surname"], inplace=True)


df_combined.dtypes
df_combined.shape

**PART 2**

**Task 6: Customer Segmentation**

**6.1 — Age-based segmentation:**

In [ ]:
bins = [0, 25, 35, 45, 55, 65, 100]
labels = ["Under 25", "25–34", "35–44", "45–54", "55–64", "65+"]

df_combined["AgeGroup"] = pd.cut(df_combined["Age"], bins=bins, labels=labels, right=False)
df_combined["AgeGroup"].value_counts()

**6.2 Value-based segmentation:**

In [ ]:
df_combined["ValueSegment"] = pd.qcut(
    df_combined["Balance"],
    q=4,
    duplicates='drop'
)
df_combined["ValueSegment"].value_counts()

**Task 7: Grouped Aggregation Analysis**

In [ ]:
# Average Balance by AgeGroup
df_combined.groupby("AgeGroup", observed=True)["Balance"].mean()

# Mean CreditScore by Churn status
df_combined.groupby("Churn", observed=True)["CreditScore"].mean()

# Churn rate by Geography
df_combined.groupby("Geography", observed=True)["Churn"].apply(
    lambda x: (x.astype(int) == 1).sum() / len(x) * 100
)

# Multi-level grouping: Churn rate by Geography AND Gender
df_combined.groupby(["Geography", "Gender"], observed=True)["Churn"].apply(
    lambda x: (x.astype(int) == 1).sum() / len(x) * 100
).reset_index(name="ChurnRate%")

# Average Balance by AgeGroup AND ValueSegment
df_combined.groupby(["AgeGroup", "ValueSegment"], observed=True)["Balance"].mean()

**Task 8: Pivot Table Construction**

In [ ]:
pivot = pd.pivot_table(
    df_combined,
    values="Churn",
    index="Geography",
    columns="Gender",
    aggfunc=lambda x: (x.astype(int) == 1).sum() / len(x) * 100,
    observed=True
)

pivot.round(2)